In [10]:
import json
import pandas as pd
from tqdm.auto import tqdm

from openai import OpenAI

In [3]:
with open("documents-with-ids.json", "rt") as f_in:
    documents = json.load(f_in)

In [4]:
documents[0]

{'text': "The purpose of this document is to capture frequently asked technical questions\nThe exact day and hour of the course will be 15th Jan 2024 at 17h00. The course will start with the first  “Office Hours'' live.1\nSubscribe to course public Google Calendar (it works from Desktop only).\nRegister before the course starts using this link.\nJoin the course Telegram channel with announcements.\nDon’t forget to register in DataTalks.Club's Slack and join the channel.",
 'section': 'General course-related questions',
 'question': 'Course - When will the course start?',
 'course': 'data-engineering-zoomcamp',
 'id': 'c02e79ef'}

In [19]:
prompt_template = """
You are a student who is taking the course. Your task is to generate 5 questions that
a student would ask based on the FAQ CONTEXT. Use fewer words as possible as you can from the CONTEXT.

<CONTEXT>
SECTION: {section}
QUESTION: {question}
ANSWER: {text}
</CONTEXT>

Produce the output in parsable JSON format as below. Don't use code blocks.
["question1", "question2", ..., "question5"]
""".strip()

In [20]:
client = OpenAI()

def generate_questions(doc):
    prompt = prompt_template.format(**doc)

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}]
    )

    return response.choices[0].message.content

In [23]:
results = {}

for doc in tqdm(documents[:3]):
    doc_id = doc["id"]
    if doc_id in results:
        continue

    questions = generate_questions(doc)
    results[doc_id] = questions

  0%|          | 0/3 [00:00<?, ?it/s]

In [29]:
results[next(iter(results))]

'["When does the course begin?", "What time does the course start?", "How can I register for the course?", "Is there a calendar for course dates?", "Where can I find course announcements?"]'

In [31]:
parsed_results = {}

for doc_id, questions in results.items():
    parsed_results[doc_id] = json.loads(questions)

In [33]:
parsed_results[next(iter(results))]

['When does the course begin?',
 'What time does the course start?',
 'How can I register for the course?',
 'Is there a calendar for course dates?',
 'Where can I find course announcements?']

In [34]:
doc_idx = {d["id"]: d for d in documents}

In [35]:
final_results = []

for doc_id, questions in parsed_results.items():
    course = doc_idx[doc_id]["course"]
    for q in questions:
        final_results.append((q, course, doc_id))

In [37]:
df_ground_truth = pd.DataFrame(final_results, columns=["question", "course", "document"])

In [38]:
df_ground_truth

,question,course,document
0,When does the course begin?,data-engineering-zoomcamp,c02e79ef
1,What time does the course start?,data-engineering-zoomcamp,c02e79ef
2,How can I register for the course?,data-engineering-zoomcamp,c02e79ef
3,Is there a calendar for course dates?,data-engineering-zoomcamp,c02e79ef
4,Where can I find course announcements?,data-engineering-zoomcamp,c02e79ef
5,What are the course prerequisites?,data-engineering-zoomcamp,1f6520ca
6,Where can I find prerequisites?,data-engineering-zoomcamp,1f6520ca
7,Are there any prerequisites?,data-engineering-zoomcamp,1f6520ca
8,Prerequisites for this course?,data-engineering-zoomcamp,1f6520ca
9,Link to course prerequisites?,data-engineering-zoomcamp,1f6520ca
